# Hybrid DeepSeek

This notebook evaluates hybrid emotion classification methods by combining:

- Machine Learning baseline
- Deep Learning baseline
- DeepSeek local model

The experiments include:

1. ML + DeepSeek
2. DL + DeepSeek
3. ML + DL + DeepSeek

### Setup 

In [1]:
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import joblib

from openai import OpenAI

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

### Path Configuration

In [2]:
cwd = Path.cwd().resolve()

candidates = [cwd, cwd.parent, cwd.parent.parent]

BASE_DIR = None
for c in candidates:
    if c.name == "Senior_Project_Text_Mining" and (c / "data").exists():
        BASE_DIR = c
        break

if BASE_DIR is None:
    raise FileNotFoundError(f"Cannot find project root from cwd={cwd}")

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"

print("BASE_DIR:", BASE_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)

BASE_DIR: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining
PROCESSED_DIR: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining\data\processed
MODELS_DIR: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining\models


## Load Processed Dataset
This section loads the cleaned dataset used for hybrid experiments.

In [3]:
df = pd.read_csv(PROCESSED_DIR / "cleaned_emotions_dl.csv")

df = df.rename(columns={"sentence": "text"})

print(df.shape)
print(df.columns.tolist())
df.head()

(2160, 2)
['text', 'label']


,text,label
0,i am not calm about checking the swollen gum t...,fear
1,i keep thinking about how long i've been livin...,disgust
2,this whole situation makes me angry because i ...,angry
3,i am so happy everything looks better now,happy
4,it's been leaking that disgusting taste down m...,disgust


### Label Distribution

In [4]:
print(df["label"].value_counts())

label
neutral    500
disgust    427
fear       395
angry      326
sad        278
happy      234
Name: count, dtype: int64


## Prepare Train, Validation, and Test Split
This section creates consistent data splits for evaluation.

### Train/Validation/Test Split

In [5]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Train: (1512, 2)
Val  : (324, 2)
Test : (324, 2)


### Prepare Text and Labels

In [6]:
X_train_text = train_df["text"].astype(str)
y_train_text = train_df["label"].astype(str)

X_val_text = val_df["text"].astype(str)
y_val_text = val_df["label"].astype(str)

X_test_text = test_df["text"].astype(str)
y_test_text = test_df["label"].astype(str)

## Build ML Baseline for Hybrid Use
This section trains a TF-IDF + Logistic Regression model for hybrid experiments.

### Train ML Baseline

In [7]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=1
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_val_tfidf = vectorizer.transform(X_val_text)
X_test_tfidf = vectorizer.transform(X_test_text)

ml_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

ml_model.fit(X_train_tfidf, y_train_text)

val_pred_ml = ml_model.predict(X_val_tfidf)

print("ML validation accuracy:", accuracy_score(y_val_text, val_pred_ml))
print(classification_report(y_val_text, val_pred_ml))

ML validation accuracy: 0.904320987654321
              precision    recall  f1-score   support

       angry       0.98      1.00      0.99        49
     disgust       0.84      0.91      0.87        64
        fear       0.88      0.75      0.81        59
       happy       1.00      0.97      0.99        35
     neutral       0.84      0.88      0.86        75
         sad       1.00      1.00      1.00        42

    accuracy                           0.90       324
   macro avg       0.92      0.92      0.92       324
weighted avg       0.91      0.90      0.90       324



## Load DL Baseline Model
This section loads the pretrained BiLSTM model.

In [8]:
dl_model = load_model(MODELS_DIR / "bilstm_model.keras")
print("DL model loaded successfully")

DL model loaded successfully


## Reconstruct Tokenizer and Label Mapping
This section restores the tokenizer and label mapping used in the DL pipeline.

### Load Tokenizer Vocabulary

In [9]:
with open(PROCESSED_DIR / "word_index.json", "r", encoding="utf-8") as f:
    word_index = json.load(f)

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.word_index = word_index
tokenizer.index_word = {idx: word for word, idx in word_index.items()}

print("Tokenizer reconstructed")
print("Vocabulary size:", len(word_index))

Tokenizer reconstructed
Vocabulary size: 1346


### Load Label Mapping

In [10]:
with open(PROCESSED_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    label_mapping = json.load(f)

inverse_label_mapping = {int(v): k for k, v in label_mapping.items()}

print("label_mapping:", label_mapping)
print("inverse_label_mapping:", inverse_label_mapping)

label_mapping: {'angry': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5}
inverse_label_mapping: {0: 'angry', 1: 'disgust', 2: 'fear', 3: 'happy', 4: 'neutral', 5: 'sad'}


### Determine Maximum Sequence Length

In [11]:
X_test_pad_saved = np.load(PROCESSED_DIR / "X_test_pad.npy")
MAX_LEN = X_test_pad_saved.shape[1]

print("MAX_LEN:", MAX_LEN)

MAX_LEN: 38


## Create Prediction Functions
This section defines prediction functions for ML and DL models.

### Text Cleaning Function

In [12]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

### ML Prediction Function

In [13]:
def predict_ml(text):
    x = vectorizer.transform([text])
    pred = ml_model.predict(x)[0]
    return pred

### DL Prediction Function

In [14]:
def predict_dl(text):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob = dl_model.predict(pad, verbose=0)
    pred_id = int(np.argmax(prob, axis=1)[0])
    return inverse_label_mapping[pred_id]

### Quick Baseline Test

In [15]:
sample_text = "i am really scared about this surgery"

print("Text:", sample_text)
print("ML prediction:", predict_ml(sample_text))
print("DL prediction:", predict_dl(sample_text))

Text: i am really scared about this surgery
ML prediction: fear
DL prediction: angry


## Connect to Local DeepSeek Model
This section connects the notebook to the local DeepSeek model using Ollama.

In [16]:
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

ALLOWED_LABELS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]

print("DeepSeek local client ready")

DeepSeek local client ready


## Create DeepSeek Refinement Function
This section defines a function that uses DeepSeek to refine the final prediction.

### Normalize DeepSeek Output

In [17]:
def normalize_label(text):
    text = str(text).strip().lower()
    text = text.replace("```", " ").replace("\n", " ")

    for label in ALLOWED_LABELS:
        if re.search(rf"\b{label}\b", text):
            return label

    return "neutral"

### DeepSeek Refinement Function

In [18]:
def predict_deepseek_local(text, ml_pred, dl_pred):
    prompt = f"""
You are an emotion classifier.

Return exactly one label from this list only:
angry, disgust, fear, happy, neutral, sad

Text: {text}
ML prediction: {ml_pred}
DL prediction: {dl_pred}

Answer only one label.
"""

    res = client.chat.completions.create(
        model="deepseek-r1:8b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    raw_output = res.choices[0].message.content.strip()
    final_label = normalize_label(raw_output)

    return raw_output, final_label

## ML + DeepSeek Experiment
This section evaluates the hybrid model that combines ML with DeepSeek.

In [19]:
def hybrid_ml_deepseek(text):
    ml_pred = predict_ml(text)

    raw_output, final_pred = predict_deepseek_local(
        text=text,
        ml_pred=ml_pred,
        dl_pred=ml_pred
    )

    return {
        "text": text,
        "ml_prediction": ml_pred,
        "deepseek_raw": raw_output,
        "final_prediction": final_pred
    }

## DL + DeepSeek Experiment
This section evaluates the hybrid model that combines DL with DeepSeek.

In [20]:
def hybrid_dl_deepseek(text):
    dl_pred = predict_dl(text)

    raw_output, final_pred = predict_deepseek_local(
        text=text,
        ml_pred=dl_pred,
        dl_pred=dl_pred
    )

    return {
        "text": text,
        "dl_prediction": dl_pred,
        "deepseek_raw": raw_output,
        "final_prediction": final_pred
    }

## Full Hybrid Experiment: ML + DL + DeepSeek
This section evaluates the full hybrid model using both ML and DL predictions with DeepSeek.

In [21]:
def hybrid_all(text):
    ml_pred = predict_ml(text)
    dl_pred = predict_dl(text)

    raw_output, final_pred = predict_deepseek_local(
        text=text,
        ml_pred=ml_pred,
        dl_pred=dl_pred
    )

    return {
        "text": text,
        "ml_prediction": ml_pred,
        "dl_prediction": dl_pred,
        "deepseek_raw": raw_output,
        "final_prediction": final_pred
    }

### Quick Test for All Hybrid Models

In [22]:
text = "i am really scared about this surgery"

print("ML + DeepSeek:")
print(hybrid_ml_deepseek(text))

print("\nDL + DeepSeek:")
print(hybrid_dl_deepseek(text))

print("\nML + DL + DeepSeek:")
print(hybrid_all(text))

ML + DeepSeek:
{'text': 'i am really scared about this surgery', 'ml_prediction': 'fear', 'deepseek_raw': 'fear', 'final_prediction': 'fear'}

DL + DeepSeek:
{'text': 'i am really scared about this surgery', 'dl_prediction': 'angry', 'deepseek_raw': 'fear', 'final_prediction': 'fear'}

ML + DL + DeepSeek:
{'text': 'i am really scared about this surgery', 'ml_prediction': 'fear', 'dl_prediction': 'angry', 'deepseek_raw': 'fear', 'final_prediction': 'fear'}


## Compare All Models
This section compares ML, DL, ML+DeepSeek, DL+DeepSeek, and the full hybrid model.

### Evaluate All Models on Sample Test Data

In [23]:
eval_df = test_df.sample(min(30, len(test_df)), random_state=42).copy()

results = []

for text, true_label in zip(eval_df["text"].astype(str), eval_df["label"].astype(str)):
    ml_pred = predict_ml(text)
    dl_pred = predict_dl(text)
    ml_ds = hybrid_ml_deepseek(text)["final_prediction"]
    dl_ds = hybrid_dl_deepseek(text)["final_prediction"]
    all_ds = hybrid_all(text)["final_prediction"]

    results.append({
        "text": text,
        "true_label": true_label,
        "ml_pred": ml_pred,
        "dl_pred": dl_pred,
        "ml_deepseek_pred": ml_ds,
        "dl_deepseek_pred": dl_ds,
        "all_hybrid_pred": all_ds
    })

results_df = pd.DataFrame(results)
results_df.head()

,text,true_label,ml_pred,dl_pred,ml_deepseek_pred,dl_deepseek_pred,all_hybrid_pred
0,can you tell if i have been grinding on that s...,neutral,neutral,neutral,neutral,neutral,neutral
1,i'm about to go to a different clinic if this ...,angry,angry,angry,angry,angry,angry
2,i've been waiting for an hour this is ridiculous,angry,angry,angry,angry,angry,angry
3,i keep brushing harder but nothing changes and...,disgust,disgust,disgust,disgust,disgust,disgust
4,i am not only hurting physically having to dea...,sad,sad,sad,sad,sad,sad


### Accuracy Comparison

In [24]:
summary_rows = [
    {
        "Model": "ML",
        "Accuracy": accuracy_score(results_df["true_label"], results_df["ml_pred"]),
        "F1_Macro": f1_score(results_df["true_label"], results_df["ml_pred"], average="macro")
    },
    {
        "Model": "DL",
        "Accuracy": accuracy_score(results_df["true_label"], results_df["dl_pred"]),
        "F1_Macro": f1_score(results_df["true_label"], results_df["dl_pred"], average="macro")
    },
    {
        "Model": "ML + DeepSeek",
        "Accuracy": accuracy_score(results_df["true_label"], results_df["ml_deepseek_pred"]),
        "F1_Macro": f1_score(results_df["true_label"], results_df["ml_deepseek_pred"], average="macro")
    },
    {
        "Model": "DL + DeepSeek",
        "Accuracy": accuracy_score(results_df["true_label"], results_df["dl_deepseek_pred"]),
        "F1_Macro": f1_score(results_df["true_label"], results_df["dl_deepseek_pred"], average="macro")
    },
    {
        "Model": "ML + DL + DeepSeek",
        "Accuracy": accuracy_score(results_df["true_label"], results_df["all_hybrid_pred"]),
        "F1_Macro": f1_score(results_df["true_label"], results_df["all_hybrid_pred"], average="macro")
    }
]

summary_df = pd.DataFrame(summary_rows).sort_values(by="F1_Macro", ascending=False)
summary_df

,Model,Accuracy,F1_Macro
1,DL,0.866667,0.881481
3,DL + DeepSeek,0.866667,0.871678
0,ML,0.866667,0.871591
2,ML + DeepSeek,0.866667,0.871591
4,ML + DL + DeepSeek,0.866667,0.870667


In [25]:
output_path = PROCESSED_DIR / "hybrid_comparison_results.csv"
results_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)

Saved to: C:\3rd Year\Senior Project\Emotion_Analysis\Senior_Project_Text_Mining\data\processed\hybrid_comparison_results.csv
